# Notebook 4: Advanced Forecasting Models

## Overview

This notebook estimates two modern forecasting approaches:

1. **Facebook Prophet** — additive decomposition model with flexible trend changepoints and 95% prediction intervals derived from an uncertainty model over trend, seasonality, and observation noise.

2. **LSTM Neural Network** — two-layer LSTM (64/32 units) with MC-Dropout (200 forward passes) for uncertainty quantification and 95% prediction intervals.

Both models are fitted on the 1990–2018 training period and evaluated on 2019–2023.

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from src.data_processing import (
    MENA_COUNTRIES, get_country_series, train_test_split_series,
    prepare_prophet_df, normalize_series, create_lstm_sequences,
)
from src.models import (
    fit_prophet, forecast_prophet,
    build_lstm, train_lstm, forecast_lstm,
)
from src.evaluation import compute_all_metrics
from src.visualization import plot_forecast

panel = pd.read_csv('../data/processed/mena_gdp_panel.csv')
LOOKBACK = 5
STEPS = 5

## Facebook Prophet

Prophet decomposes the time series into trend, seasonality, and holidays. For annual GDP growth, yearly seasonality is enabled and changepoint flexibility is set to 0.05 (the default). The model provides native 95% prediction intervals via uncertainty sampling.


In [ ]:
prophet_models = {}
prophet_forecasts = {}
prophet_metrics = {}

for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)

    train_df = prepare_prophet_df(train)

    model = fit_prophet(train_df)
    prophet_models[code] = model

    fdf = forecast_prophet(model, steps=len(test))
    prophet_forecasts[code] = fdf

    metrics = compute_all_metrics(
        test.values, fdf['forecast'].values, train.values
    )
    prophet_metrics[code] = metrics
    print(f'{name:20s}  MAE={metrics["MAE"]:.2f}%  RMSE={metrics["RMSE"]:.2f}%')

In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)
    plot_forecast(
        train, test, prophet_forecasts[code],
        model_name='Prophet', country_name=name,
        save_path=f'../results/figures/04_prophet_forecast_{code}.png',
    )

## LSTM Neural Network

The LSTM is trained country-by-country on normalised sequences of length 5. Architecture: two stacked LSTM layers (64 and 32 units) with 20% dropout after each. Training uses the Adam optimiser with early stopping (patience = 20 epochs, maximum 200 epochs). Uncertainty quantification employs MC-Dropout: dropout is kept active at inference and 200 forward passes are sampled to construct empirical 95% prediction intervals.


In [ ]:
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

lstm_forecasts = {}
lstm_metrics = {}

for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)

    train_scaled, test_scaled, scaler = normalize_series(train, test)

    X_train, y_train = create_lstm_sequences(train_scaled, lookback=LOOKBACK)

    model = build_lstm(lookback=LOOKBACK)
    train_lstm(model, X_train, y_train)

    last_seq = train_scaled[-LOOKBACK:]
    fdf = forecast_lstm(model, last_seq, steps=STEPS, scaler=scaler)
    lstm_forecasts[code] = fdf

    metrics = compute_all_metrics(
        test.values, fdf['forecast'].values, train.values
    )
    lstm_metrics[code] = metrics
    print(f'{name:20s}  MAE={metrics["MAE"]:.2f}%  RMSE={metrics["RMSE"]:.2f}%')

In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    train, test = train_test_split_series(series)
    plot_forecast(
        train, test, lstm_forecasts[code],
        model_name='LSTM', country_name=name,
        save_path=f'../results/figures/04_lstm_forecast_{code}.png',
    )

## Save Advanced Model Results


In [ ]:
for code in MENA_COUNTRIES:
    prophet_forecasts[code].to_csv(f'../results/tables/prophet_forecast_{code}.csv', index=False)
    lstm_forecasts[code].to_csv(f'../results/tables/lstm_forecast_{code}.csv', index=False)

pd.DataFrame(prophet_metrics).T.to_csv('../results/tables/prophet_metrics.csv')
pd.DataFrame(lstm_metrics).T.to_csv('../results/tables/lstm_metrics.csv')

print('Advanced model results saved.')

## Summary

Prophet's additive decomposition with flexible changepoints captures the regime shifts in MENA GDP growth trajectories more effectively than the classical ARIMA/ETS specifications. The LSTM benefits from learning non-linear dependencies but is constrained by the relatively short per-country series (29 training observations). Notebook 5 provides the formal cross-model comparison and statistical tests.
